# L13b: Interpretable Allocation and Model-Predictive Rebalancing
Previously, we chose the share counts of a portfolio once, by maximizing a utility function under a budget. How do we manage that portfolio every day, and how do we judge it when the real market gives us only one path?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Pose the allocation problem with trading limits:__ Write the maximum-utility problem over share counts with its budget and its trading limits. Explain how the score of a stock combines its expected growth rate from the single index model with the market state.
> * __Explain when a model-predictive controller re-plans:__ Describe the forecast band of a geometric Brownian motion forecast and the three triggers, which are the horizon, a band exit, and a drawdown stop. Read a real-data backtest that compares selection rules and weighting rules.
> * __Evaluate a trading method beyond the historical record:__ Explain why a backtest is one historical path. Describe how typical and stressed synthetic markets from a jump hidden Markov model test the same method.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
> * Each rebalance chooses the shares $n_{i}$ of each stock to maximize a utility $U(n;\gamma)$ within a budget and four trading limits: a single-name cap, a volatility cap, a turnover budget, and, with the hedge on, a $\beta$ cap. The score $\gamma_{i}=\tanh\left((\alpha_{i}+\beta_{i}\,\tilde{g}_{M})/\max(|\beta_{i}|,0.1)^{\xi}\right)$ has the sign of the stock's expected growth rate, and the market state $\xi$ changes only its size.
> * The book re-plans when 21 trading days have passed, when its wealth leaves the 95% band of a forecast, or when it falls 8% below its peak. Between re-plans it does not trade.
> * A backtest is one historical path. A market generator fitted to the same stocks gives many typical futures and sell-offs on which to test the same method.
>
> __For understanding__ (you will not need to reproduce or memorize these):
> * The four utilities and their solutions under the budget alone, the three benchmark weighting rules, and the hedge pool.
> * The daily update of the single index model, the covariance of the forecast, and the trading costs.
> * The ten selection rules, and how the typical and stress sets are chosen from 10,000 simulated paths.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13b) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples
__We will work through these examples in lecture:__

> [▶ Run the INFORMS portfolio method on real data](CHEME-5660-L13b-Example-MPC-RealData-Fall-2026.ipynb). In this example, we calibrate the method on 2014 to 2024 for the 373 S&P 500 stocks with a complete history and trade 2025 and 2026. We follow the market state and the scores, run one book day by day and read why it re-planned, and then change the weighting rule, the selection rule, the hedge, and the basket on the same path.

> [▶ Run the INFORMS portfolio method on synthetic markets](CHEME-5660-L13b-Example-MPC-SyntheticData-Fall-2026.ipynb). In this example, we draw synthetic markets from a generator fitted to the same 373 stocks, and choose typical markets and two kinds of sell-off as the study does. We then run the method on each market, and compare the books with `SPY` and an equal-weight portfolio set by set.

___

## Live Portfolio Management
Live portfolio management couples three problems: allocation, the timing of rebalancing, and evaluation beyond the historical record.

* __Allocation (what to hold):__ At each rebalance, we maximize a utility of a score $\gamma_{i}$ for each stock, within trading limits. The score comes from the single index model and the recent behavior of the market.
* __Rebalancing (when to trade):__ We re-plan at a 21-day horizon, at a band exit, or at a drawdown stop. We test these rules with a real-data backtest from January 2025 to September 2026.
* __Market generator (evaluation):__ A backtest is one historical path. A jump hidden Markov model (JumpHMM) generates typical and stressed futures for the same stocks.

This lecture follows a study presented at the INFORMS Annual Meeting 2026: J. D. Varner, "Interpretable Allocation and Model-Predictive Rebalancing of Equity Portfolios, Tested on Real and Synthetic Markets", [arXiv:2611.03022](https://arxiv.org/abs/2611.03022). Its code, results, and slides are in a [public repository](https://github.com/varnerlab/INFORMS-Annual-Meeting-2026-Varnerlab).

The study uses daily data for the 373 S&P 500 stocks that have a complete history from 2014 to 2026 and a sector. It calibrates on 2014 to 2024 and trades from January 2, 2025 to September 4, 2026, which is 420 trading days. Every decision uses data dated on or before the day it is made.
___

## Allocation
What should the book hold? A __selection rule__, which the examples call a picker, picks a basket $\mathcal{N}$ of $K$ stocks from the universe. A __weighting rule__ chooses the number of shares $n_{i}\geq 0$ of each stock $i$ in the basket.

### Maximize utility subject to trading limits
Let $S_{i}$ denote the close price of stock $i$ (units: USD/share) and $W$ the wealth of the book (units: USD). The weight of stock $i$ is $w_{i}=n_{i}S_{i}/W$.

Each rebalance chooses the shares $n=(n_{1},\dots,n_{K})$ to solve:

$$
\boxed{\begin{aligned}
\operatorname*{maximize}_{n\,\geq\,0}\quad & U(n;\gamma) &&\quad\text{(utility)}\\
\text{subject to}\quad & \sum_{i\in\mathcal{N}}n_{i}S_{i}\leq W &&\quad\text{(budget)}\\
& w_{i}\leq\bar{w} &&\quad\text{(single-name cap)}\\
& \sqrt{w^{\top}\Sigma_{g}\,w}\leq\sigma_{\text{SPY}} &&\quad\text{(volatility cap)}\\
& \bar{c}\,\lVert n-n^{\text{prev}}\rVert_{1}\leq B &&\quad\text{(turnover budget)}\\
& \beta^{\top}w\leq\bar{\beta} &&\quad\text{(}\beta\text{ cap, hedge on)}
\end{aligned}}
$$

where $\gamma_{i}$ is the score of stock $i$, which we define in the next section. Wealth that the budget leaves unspent stays in cash. The four trading limits are:

| Limit | What it does | Setting in the study |
|:--|:--|:--|
| Single-name cap $\bar{w}$ | No stock is more than the fraction $\bar{w}$ of the wealth. | $\bar{w}=\max(0.10,\,1.5/K)$, which is 11.5% for 13 stocks and 10% for 33 |
| Volatility cap $\sigma_{\text{SPY}}$ | The book is no more volatile than `SPY`. The matrix $\Sigma_{g}$ is the covariance of the stocks' daily growth rates (units: 1/yr²). | The standard deviation of the daily growth rate of `SPY` over 2014 to 2024, a volatility of 17.3% a year |
| Turnover budget $B$ | A rebalance spends at most $B$ on the bid–ask spread. The vector $n^{\text{prev}}$ holds the shares before the rebalance, and $\bar{c}$ is the half spread per share (units: USD/share). | $B=0.10\,W$, with a half spread of 5 basis points of the mean close price |
| $\beta$ cap $\bar{\beta}$ | The book's $\beta$ to `SPY` is at most $\bar{\beta}$, where $\beta_{i}$ is the single index model $\beta$ of stock $i$. | $\bar{\beta}=\max(0,\,1-\xi)$, where $\xi$ is the market state defined in the next section |

The $\beta$ cap applies only when the __market hedge__ is on. The cap equals 1 when $\xi=0$, and it tightens toward zero as the market state turns bearish ($\xi>0$). With the hedge on, the book can also hold up to five stocks whose $\beta$ is zero or negative, the __hedge pool__.

The study compares four utilities. Each is a function of the stocks in the basket with a positive score, $\gamma_{i}>0$:

| Utility | $U(n;\gamma)$ | Solution with the budget alone |
|:--|:--|:--|
| Cobb–Douglas | $\sum_{i}\gamma_{i}\ln n_{i}$ | $w_{i}\propto\gamma_{i}$ |
| CES on shares | $\left(\sum_{i}\gamma_{i}\,n_{i}^{\rho}\right)^{1/\rho}$ | $n_{i}\propto\left(\gamma_{i}/S_{i}\right)^{\eta}$ |
| CES on weights | $\left(\sum_{i}\gamma_{i}\,w_{i}^{\rho}\right)^{1/\rho}$ | $w_{i}\propto\gamma_{i}^{\eta}$ |
| Stone–Geary | $\sum_{i}\gamma_{i}\ln\left(n_{i}-b_{i}\right)$ | $\left(n_{i}-b_{i}\right)S_{i}\propto\gamma_{i}$ |

where $\rho=1-1/\eta$ for the elasticity $\eta=0.5$, and the Stone–Geary baseline $b_{i}$ is the number of shares worth 1% of the wealth, $b_{i}S_{i}=0.01\,W$. The first entry is the logarithm of the Cobb–Douglas utility $\prod_{i}n_{i}^{\gamma_{i}}$, which has the same maximizer.

The last column gives the solution when the budget is the only constraint, the closed forms we derived previously. With the trading limits, there is no closed form. Each problem is a convex cone program, and a numerical solver finds its maximum.

The study also runs three benchmarks under the same limits: minimum variance, the tangency portfolio, and a 130/30 long–short book. Together, the four utilities and the three benchmarks are the seven weighting rules.


### The score γ: expected growth, tilted by the market
Each stock gets a score after every close. The score of stock $i$ is given by:

$$
\boxed{\gamma_{i}=\tanh\left(\frac{\alpha_{i}+\beta_{i}\,\tilde{g}_{M}}{\max\left(|\beta_{i}|,\,0.1\right)^{\xi}}\right)}
$$

where $(\alpha_{i},\beta_{i})$ are the single index model parameters of stock $i$ against `SPY`, updated after every close. The market growth rate $\tilde{g}_{M}$ is the exponential moving average (EMA) of the daily growth rate of `SPY` over 252 trading days (units: 1/yr).

The numerator $\alpha_{i}+\beta_{i}\,\tilde{g}_{M}$ is the __expected growth rate__ of stock $i$ (units: 1/yr). The denominator is the __$\beta$ penalty__, which depends on the __market state__ $\xi$. The market state compares the 21-day and the 63-day EMA of the `SPY` close:

$$
\xi=-G\left(\frac{\text{EMA}_{21}}{\text{EMA}_{63}}-1\right)
$$

where the gain $G=21.23$ is chosen so that $|\xi|\leq 1$ on 99% of the days from 2014 to 2024. The market state is positive when $\text{EMA}_{21}<\text{EMA}_{63}$, as in a falling market, and negative in a rising one.

Let's look at the market state over the trading window:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-MarketState.svg" width="900" style="max-width:100%; height:auto;" alt="Two strips share a date axis from January 2025 to September 2026. The top strip shows the SPY close in gold with its 21-day exponential moving average in black and its 63-day exponential moving average in blue, each divided by the first SPY close. It rises from 1.0 to about 1.3, with a dip to about 0.9 in April 2025. The bottom strip shows the market state, which stays between minus 0.8 and 0 for most of the window and rises above zero twice, to about 0.9 in April 2025 and to about 0.4 in March 2026. Both strips shade the days on which the market state is positive, from March to May 2025 and from March to April 2026. Four open rings mark the days on which the two moving averages cross, and a dotted line runs from each ring down to the point where the market state crosses zero."/>
    </center>
</div>

__The market state, January 2025 to September 2026.__ The top strip shows `SPY` and its two EMAs, each divided by the first `SPY` close. The bottom strip shows $\xi$. Each ring marks a day on which the two EMAs cross, where $\xi=0$.

The shaded days have $\xi>0$. The market state was positive on 75 of the 420 trading days, in the sell-off of spring 2025 and in March and April 2026. It ranged from −0.80 to 0.90.

__How do we read the score?__ The score has three parts:

* __Sign:__ The $\beta$ penalty is positive, so the score has the sign of the expected growth rate. A stock that the single index model expects to grow has a positive score.
* __Size:__ In a falling market ($\xi>0$), the expected growth rate of a stock with $|\beta_{i}|>1$ is divided by more than one, and that of a stock with $|\beta_{i}|<1$ by less than one. The score then favors stocks with a low $\beta$. In a rising market ($\xi<0$), it favors stocks with a high $\beta$.
* __Bounds:__ The hyperbolic tangent keeps the score between −1 and 1. The floor of 0.1 keeps the $\beta$ penalty finite for a stock whose $\beta_{i}$ is near zero.

This score is the preference weight we used previously, in a form that works for every stock. Previously, we assumed $\beta_{i}>0$. Here, $\beta_{i}$ keeps its sign in the numerator, and a stock with $\beta_{i}<0$ can join the hedge pool.

The score does two jobs. For __weighting__, the score enters the utility $U(n;\gamma)$: stocks with $\gamma_{i}>0$ are held long, and the 130/30 rule shorts stocks with $\gamma_{i}<0$. For __selection__, a rule picks the stocks with the highest scores, such as the 13 highest overall or the 3 highest in each sector.
___

## Rebalancing
When should the book trade? A rebalance sets target shares, and the plan is to hold them. We call each rebalance a __re-plan__.

Trading every day costs money, and trading on a fixed calendar ignores what the market does in between.

### Rebalancing triggers: horizon, forecast band, drawdown
The method uses __model-predictive control__. A controller plans with the forecast of a model, measures what the real system (the __plant__) does, and re-plans when the measurement departs from the forecast. Here, the plant is the market, and the measurement is the wealth of the book at the real close prices.

At each re-plan, the controller forecasts the wealth of the book over the next 21 trading days with a multi-asset geometric Brownian motion. It simulates 1,000 paths of the close prices. On each path, the daily growth rates of the stocks have mean $\alpha_{i}+\beta_{i}\,\tilde{g}_{M}$ and covariance $\Sigma_{g}$, the same inputs that the allocation uses.

The forecast wealth on a path is the target shares times the simulated prices, plus the leftover cash growing at the 3-month Treasury bill rate. Let $\mu_{W}(\tau)$ and $\sigma_{W}(\tau)$ denote the mean and the standard deviation of the forecast wealth over the paths, $\tau$ trading days after the re-plan (units: USD).

The 95% __forecast band__ for the wealth $W$ is given by:

$$
\mu_{W}(\tau)-1.96\,\sigma_{W}(\tau)\;\leq\;W\;\leq\;\mu_{W}(\tau)+1.96\,\sigma_{W}(\tau)
$$

After each close, the controller checks three triggers in order, and it re-plans on the first trigger that holds:

* __Drawdown stop:__ The wealth is more than 8% below its running peak. A drawdown re-plan resets the peak to the current wealth, so the stop fires once for each fresh 8% loss.
* __Horizon:__ The plan has run for 21 trading days. The forecast ends there, and the next forecast starts from the measured wealth.
* __Band exit:__ The wealth has left the 95% band of the forecast. The book has gained or lost more than the model expected.

A re-plan solves the same allocation problem from the shares held, with that day's estimates. No trigger changes the utility or the limits. Between re-plans, the book does not trade.

Let's look at one episode of the real-data backtest:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-ForecastBand.svg" width="860" style="max-width:100%; height:auto;" alt="The wealth of one book divided by its starting wealth, in red, over 50 trading days from October 10 to December 22, 2025. It rises from 1.01 to about 1.15. Three blue fans of forecast bands, each with a 68 percent, a 95 percent, and a 99 percent band, start at October 10, November 10, and November 20 and widen with time. Gold rings mark three re-plans. The first, on November 10, is labeled 21 days passed. The second, on November 20, is labeled left the 95 percent band, 2.1 standard deviations below the mean. The third, on December 22, is labeled 21 days passed."/>
    </center>
</div>

__One episode, October 10 to December 22, 2025.__ The red line is the wealth of the book divided by its starting wealth $W_{0}$. The blue bands are the 68%, 95%, and 99% bands of three forecasts, and each gold ring is a re-plan.

The first re-plan came at the horizon. Eight trading days later, the wealth was 2.1 standard deviations below the mean of the forecast, outside the 95% band, and the book re-planned again. The third re-plan came at the next horizon.

This book holds 13 stocks picked by hand in advance, with the Cobb–Douglas utility and the hedge off. Over the whole trading window, it re-planned 27 times: 17 times at the horizon, 7 times at a band exit, and 3 times at a drawdown stop.

__Do the triggers beat a fixed schedule?__ We reran everything on a fixed 21-day schedule. The ending wealth is the same. The triggers take about a point off the worst drawdown, on real prices and in simulated sell-offs.

__What does a trade cost?__ Orders fill at the next open, sells first and then buys. Each trade pays a half spread of 5 basis points, plus an impact cost that grows with the fraction of the stock's mean daily volume that the order takes. There is no commission.


### Stock selection matters more than the weighting rule
We ran the method on the real data from January 2025 to September 2026. Each run pairs a selection rule with a weighting rule, with the hedge on or off. The study compares ten selection rules in three groups. The figure below calls a stock a name:

* __33 stocks, 3 per sector:__ In each of the 11 sectors, the rule takes the 3 highest scores, the top 3 stocks of a bandit, or 3 stocks at random. The bandit learns which stocks in a sector grew faster than `SPY` over the trailing year.
* __13 stocks:__ The global rule takes the 13 highest scores in the universe. The correlation-aware rule skips a stock whose growth rates have a correlation above 0.7 with a stock it has already accepted, and the liquid rule ranks only the 100 most traded stocks. Two random rules draw 13 stocks from those 100 or from the whole universe.
* __Hand-picked:__ The 13 stocks AAPL, MSFT, INTC, MU, AMD, NVDA, GS, BAC, WFC, C, F, GM, and JNJ were chosen in advance, without 2025 data, and are never re-picked. A second list adds WMT and PG.

Every other rule picks on the first day, every 21 trading days after that, and at every band exit. Let's look at the terminal wealth of every pair:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-RealData-Selection.svg" width="880" style="max-width:100%; height:auto;" alt="A dot plot of terminal wealth divided by initial wealth, from 1.0 to 1.7, with one row for each of ten selection rules in three groups. The group of 33 stocks, 3 per sector, has the rows gamma per sector, bandit per sector, and random per sector, and its dots lie between about 1.07 and 1.29. The group of 13 stocks has the rows global gamma, correlation-aware gamma, liquid then gamma, random from the liquid 100, and random from the whole universe, and its dots lie between about 0.99 and 1.59. The hand-picked group has the rows hand-picked 13 and hand-picked 13 plus WMT and PG, and its dots lie between about 1.16 and 1.70. Each row holds seven filled dots for the weighting rules with the hedge off and seven open dots with the hedge on. A dashed gold line marks SPY bought and held at 1.32, and a dotted line marks the equal-weight universe at 1.30."/>
    </center>
</div>

__Terminal wealth on real data.__ Each row is a selection rule, and each dot is a weighting rule. A dot is the median, over 30 runs, of the terminal wealth divided by the initial wealth. Filled dots have the hedge off, and open dots have it on.

The dashed line is `SPY` bought and held, which ended at 1.32 times the initial wealth. The dotted line is an equal-weight portfolio of all 373 stocks, which ended at 1.30.

__What do we see?__ The dots in a row are close together, and the rows are far apart. With the hedge off, the seven weighting rules on the 33 stocks picked by score ended between 1.09 and 1.23 times the initial wealth. Different baskets ended between 1.03 and 1.70, and the highest was the hand-picked list.

> __Example:__
>
> [▶ Run the INFORMS portfolio method on real data](CHEME-5660-L13b-Example-MPC-RealData-Fall-2026.ipynb). In this example, we calibrate the method on 2014 to 2024 for the 373 S&P 500 stocks with a complete history and trade 2025 and 2026. We follow the market state and the scores, run one book day by day and read why it re-planned, and then change the weighting rule, the selection rule, the hedge, and the basket on the same path.

The example runs one book at a time on this path. The backtest is one path, however, and one path cannot tell us whether a rule was good or lucky.
___

## Market Generator
How do we judge the method beyond the historical record? On real data, the method runs once, on one sequence of prices with one broad sell-off in the spring of 2025. A market generator gives us many other futures for the same stocks.

### JumpHMM: a market generator with fat tails
Previously, we built a market generator from jump hidden Markov models (JumpHMM). The daily growth rate of synthetic stock $i$ in sector $s$ combines a market draw, a sector draw, and a draw of its own:

$$
g_{i}(t)=\underbrace{\alpha_{i}}_{\text{trend}}+\underbrace{\beta_{i}(t)\,g_{m}(t)}_{\text{market}}+\underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector}}+\underbrace{\psi_{i}\,\tilde{\varepsilon}_{i}(t)}_{\text{own}}
$$

where the market draw $g_{m}(t)$ comes from one JumpHMM of `SPY`, the sector draw $f_{s}(t)$ from one JumpHMM for each of the 11 sectors, and the own draw $\tilde{\varepsilon}_{i}(t)$ from one JumpHMM for each of the 373 stocks. The loadings $\delta_{i}$ and $\psi_{i}$ scale the sector and own draws. In this study, each $\beta_{i}(t)$ drifts around the stock's estimate.

The generator is fitted to the same 373 stocks on 2014 to 2024. It reproduces the fat tails of real growth rates, but its volatility clustering fades after a day.

The controller still forecasts with a geometric Brownian motion. The test markets have fat tails that the controller's forecast model does not have.

### Simulated sell-offs: every median ends above SPY
Each synthetic market covers 2014 to 2026 on the real calendar. On each market, the method calibrates on 2014 to 2024 and trades January 2025 to September 2026 with the real settings.

We draw 10,000 paths of the market model, and rank them by the maximum drawdown of synthetic `SPY` in the trading window. Three sets of futures come from these paths:

* __Typical:__ The typical set is 600 paths drawn at random. The median drawdown of `SPY` in the trading window is 17.6%.
* __Burst stress:__ This set is the 200 deepest drawdowns among the paths with a jump burst in the trading window. A burst window has months of large moves of both signs, and the median drawdown of `SPY` is 36.4%.
* __Bear stress:__ This set is the 200 deepest drawdowns among the paths without a jump burst. These declines are slower and more sustained, and the median drawdown of `SPY` is 40.6%.

Some paths are in more than one set, so the three sets hold 978 markets. Let's look at the terminal wealth in each set:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-Synthetic-Sets.svg" width="940" style="max-width:100%; height:auto;" alt="Three dot plots side by side, titled typical, burst stress, and bear stress, share the ten selection rules of the real-data figure as rows. Each dot is the median terminal wealth divided by initial wealth across the futures of a set, for one weighting rule, with filled dots for the hedge off and open dots for the hedge on. In the typical panel the dots lie between about 1.09 and 1.24, around the dashed SPY line at 1.19 and mostly below the dotted equal-weight line at 1.23. In the burst stress panel the dots lie between about 0.92 and 1.03, all to the right of the SPY line at 0.86 and the equal-weight line at 0.92. In the bear stress panel the dots lie between about 0.77 and 1.00, all to the right of the SPY line at 0.72 and the equal-weight line at 0.75."/>
    </center>
</div>

__Terminal wealth across synthetic futures.__ Each row is a selection rule, and each dot is a weighting rule. A dot is the median, across the futures of a set, of the terminal wealth divided by the initial wealth. The dashed line is synthetic `SPY` bought and held, and the dotted line is the equal-weight portfolio.

__What do we see?__ In both stress sets, every median ended above `SPY` and above the equal-weight portfolio. The random selection rules did as well. The margin came from parts of the method that every book shares, not from stock selection or from the timing of the re-plans.

__Does selection by score help?__ In typical futures with the hedge off, 13 stocks picked by score beat 13 random stocks in 62 to 71% of the futures. In bear futures, random picks did as well or better for most weighting rules.

The generator keeps each stock's 2014 to 2024 trend, which the score estimates. Part of the advantage of the score comes from the design of the generator, and it may not carry over to real prices.

__Does the hedge pay?__ The hedge reduced terminal wealth on real prices. It added to terminal wealth in the bear stress set, whose declines were the most sustained.

> __Example:__
>
> [▶ Run the INFORMS portfolio method on synthetic markets](CHEME-5660-L13b-Example-MPC-SyntheticData-Fall-2026.ipynb). In this example, we draw synthetic markets from a generator fitted to the same 373 stocks, and choose typical markets and two kinds of sell-off as the study does. We then run the method on each market, and compare the books with `SPY` and an equal-weight portfolio set by set.

The example uses a few markets in each set, so its counts are rough. The study's figure above uses all 978 markets.
___

## Summary
In this lecture, we followed one system for managing a stock portfolio every day, from the allocation problem to its test on real and synthetic markets. We posed the allocation with trading limits, let a model-predictive controller decide when to re-plan, and judged the method on one real path and on many simulated futures.

> __Key Takeaways:__
>
> * __Allocation, stock selection mattered more than the weighting rule:__ We maximized a utility of each stock's score within a budget and trading limits, with the score built from the single index model and the market state. On real data, seven weighting rules on one basket ended close together, while different baskets ended far apart.
> * __Rebalancing, re-plans came at the horizon, a band exit, or a drawdown stop:__ We used a geometric Brownian motion forecast to put a band around the wealth of the plan, and we re-planned when the book left the band, fell 8% from its peak, or reached the 21-day horizon. About a third of the re-plans came before the horizon, and the triggers took about a point off the worst drawdown.
> * __Evaluation, in both sell-off sets every median ended above SPY:__ We tested the method on typical and stressed futures from a jump hidden Markov model generator fitted to the same stocks. The generator reproduces fat tails, but it also keeps each stock's past trend, which favors selection by score more than the real market did.

Live portfolio management couples allocation, rebalancing, and evaluation, and the study treats them as one system that we can inspect at every step. A paper-trading version of this system is running now, and its results are not part of this lecture.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___